# Phase 3 LoRA Anomaly Explainer

目标：使用 `Qwen/Qwen2.5-0.5B-Instruct` 对确定性异常事实做 LoRA SFT。模型只生成说明，不计算金额、不决定风险等级、不改变建议动作。默认只运行 smoke guard，不启动训练。

In [ ]:
from pathlib import Path
import importlib.metadata
import importlib.util
import json
import os
import random
import subprocess
import sys

def resolve_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'procureguard').exists():
            return candidate.resolve()
    raise FileNotFoundError('未找到 ProcureGuard 项目根目录')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
RUN_TRAINING = False
PREFER_UNSLOTH = True
DATA_DIR = PROJECT_ROOT / 'data' / 'phase3' / 'generated'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'phase3'
ADAPTER_DIR = ARTIFACT_DIR / 'adapters' / 'qwen2.5-0.5b-anomaly-explainer'
LOG_DIR = ARTIFACT_DIR / 'logs'
PREDICTION_DIR = ARTIFACT_DIR / 'predictions'
EVALUATION_DIR = ARTIFACT_DIR / 'evaluation'
for path in (ADAPTER_DIR, LOG_DIR, PREDICTION_DIR, EVALUATION_DIR):
    path.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print({'project_root': str(PROJECT_ROOT), 'python': sys.executable, 'run_training': RUN_TRAINING})

## Smoke Guard
检查固定数据、依赖和 GPU。`RUN_TRAINING=False` 时不会加载模型。

In [ ]:
REQUIRED_FILES = [DATA_DIR / 'train.jsonl', DATA_DIR / 'validation.jsonl', DATA_DIR / 'test.jsonl']
FALLBACK_MODULES = ['torch', 'transformers', 'datasets', 'accelerate', 'peft', 'trl']
missing_files = [str(path) for path in REQUIRED_FILES if not path.exists()]
missing_modules = [name for name in FALLBACK_MODULES if importlib.util.find_spec(name) is None]
unsloth_available = importlib.util.find_spec('unsloth') is not None
cuda_available = False
if importlib.util.find_spec('torch') is not None:
    import torch
    cuda_available = torch.cuda.is_available()
    torch.manual_seed(SEED)
    if cuda_available:
        torch.cuda.manual_seed_all(SEED)
installed_versions = {}
for package in ['torch', 'transformers', 'datasets', 'accelerate', 'peft', 'trl', 'unsloth']:
    try:
        installed_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed_versions[package] = None
guard = {
    'missing_files': missing_files,
    'missing_fallback_modules': missing_modules,
    'unsloth_available': unsloth_available,
    'cuda_available': cuda_available,
    'training_ready': not missing_files and not missing_modules and cuda_available,
    'installed_versions': installed_versions,
}
(LOG_DIR / 'environment_guard.json').write_text(json.dumps(guard, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(guard, ensure_ascii=False, indent=2))
if missing_files:
    raise RuntimeError('训练数据缺失，请先运行 Phase 3 数据生成脚本')
if RUN_TRAINING and not guard['training_ready']:
    raise RuntimeError('RUN_TRAINING=True，但依赖或 CUDA guard 未通过')

## Load And Format Dataset
训练输入只序列化 `input_facts`，目标只使用 `expected_explanation`。

In [ ]:
SYSTEM_PROMPT = (
    '你是采购审核异常说明助手。只能复述输入事实，不得计算金额、改变风险等级或建议动作，'
    '不得编造政策、审批人、单号、金额或业务事实。输出必须包含异常类型、关键事实、审核结论三段。'
)

def read_jsonl(path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def to_messages(row, include_answer=True):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': json.dumps(row['input_facts'], ensure_ascii=False, sort_keys=True)},
    ]
    if include_answer:
        messages.append({'role': 'assistant', 'content': row['expected_explanation']})
    return messages

train_rows = read_jsonl(DATA_DIR / 'train.jsonl')
validation_rows = read_jsonl(DATA_DIR / 'validation.jsonl')
test_rows = read_jsonl(DATA_DIR / 'test.jsonl')
assert (len(train_rows), len(validation_rows), len(test_rows)) == (160, 20, 20)
print({'train': len(train_rows), 'validation': len(validation_rows), 'test': len(test_rows)})

## Backend Selection
优先使用 Unsloth 的 `FastLanguageModel`；不可用时使用 Transformers + PEFT + TRL。两条路径共用数据、prompt、seed、LoRA 参数和导出目录。

In [ ]:
TRAINING_CONFIG = {
    'seed': SEED,
    'max_seq_length': 1024,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'num_train_epochs': 3,
    'learning_rate': 2e-4,
    'warmup_ratio': 0.03,
    'weight_decay': 0.01,
    'logging_steps': 5,
    'eval_strategy': 'epoch',
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
}
LORA_CONFIG = {
    'r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'bias': 'none',
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
}
BACKEND = 'unsloth' if PREFER_UNSLOTH and unsloth_available else 'transformers_peft'
(LOG_DIR / 'training_config.json').write_text(
    json.dumps({'model_id': MODEL_ID, 'backend': BACKEND, 'training': TRAINING_CONFIG, 'lora': LORA_CONFIG}, indent=2),
    encoding='utf-8',
)
print({'backend': BACKEND, 'model_id': MODEL_ID, **TRAINING_CONFIG, **LORA_CONFIG})

## Model And LoRA Setup
该单元只有在 `RUN_TRAINING=True` 时加载模型。

In [ ]:
model = tokenizer = None
if RUN_TRAINING and BACKEND == 'unsloth':
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID, max_seq_length=TRAINING_CONFIG['max_seq_length'], load_in_4bit=True
    )
    model = FastLanguageModel.get_peft_model(model, use_gradient_checkpointing='unsloth', **LORA_CONFIG)
elif RUN_TRAINING:
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map='auto', quantization_config=quantization_config)
    model = get_peft_model(model, LoraConfig(task_type='CAUSAL_LM', **LORA_CONFIG))
print('model load skipped' if not RUN_TRAINING else 'model ready')

## SFT Training And Export
真实 GPU 运行时使用 TRL `SFTTrainer`，保存 adapter checkpoint 和日志。

In [ ]:
trainer = None
if RUN_TRAINING:
    from datasets import Dataset
    from transformers import TrainingArguments
    from trl import SFTTrainer
    def format_row(row):
        return {'text': tokenizer.apply_chat_template(to_messages(row), tokenize=False)}
    train_dataset = Dataset.from_list([format_row(row) for row in train_rows])
    validation_dataset = Dataset.from_list([format_row(row) for row in validation_rows])
    arguments = TrainingArguments(
        output_dir=str(ARTIFACT_DIR / 'trainer'), report_to='none',
        per_device_eval_batch_size=2, bf16=True,
        **{key: value for key, value in TRAINING_CONFIG.items() if key not in {'max_seq_length'}},
    )
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_dataset, eval_dataset=validation_dataset,
        dataset_text_field='text', max_seq_length=TRAINING_CONFIG['max_seq_length'], args=arguments,
    )
    trainer.train()
    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    (LOG_DIR / 'trainer_log_history.json').write_text(json.dumps(trainer.state.log_history, indent=2), encoding='utf-8')
print('training skipped' if not RUN_TRAINING else f'adapter saved to {ADAPTER_DIR}')

## Base Vs Fine-tuned Inference
对同一 20 条 test split 导出 `base.jsonl` 和 `fine_tuned.jsonl`。生成参数必须一致。

In [ ]:
GENERATION_CONFIG = {'max_new_tokens': 256, 'do_sample': False}
def write_predictions(path, rows, generate_one):
    with path.open('w', encoding='utf-8', newline='\n') as handle:
        for row in rows:
            handle.write(json.dumps({'sample_id': row['sample_id'], 'explanation': generate_one(row)}, ensure_ascii=False) + '\n')

# 真实运行时分别加载原始 base model 和 ADAPTER_DIR，使用相同 tokenizer/chat template/GENERATION_CONFIG。
# write_predictions(PREDICTION_DIR / 'base.jsonl', test_rows, generate_with_base)
# write_predictions(PREDICTION_DIR / 'fine_tuned.jsonl', test_rows, generate_with_adapter)
print({'prediction_dir': str(PREDICTION_DIR), 'generation': GENERATION_CONFIG})

## Unified Evaluation
只有两份真实预测都存在时才调用统一评测脚本；否则明确跳过，不产生占位数字。

In [ ]:
base_predictions = PREDICTION_DIR / 'base.jsonl'
fine_tuned_predictions = PREDICTION_DIR / 'fine_tuned.jsonl'
if base_predictions.exists() and fine_tuned_predictions.exists():
    command = [
        sys.executable, str(PROJECT_ROOT / 'scripts' / 'phase3' / 'evaluate_explanations.py'),
        '--dataset', str(DATA_DIR / 'test.jsonl'),
        '--base-predictions', str(base_predictions),
        '--fine-tuned-predictions', str(fine_tuned_predictions),
        '--output-dir', str(EVALUATION_DIR),
    ]
    subprocess.run(command, check=True)
else:
    print('尚无真实 base/fine-tuned 推理文件，不生成评测指标。')